### Holiday calendar sensitivity

The supplied `day_calendar.csv` is joined by date and hashed with the analysis inputs. It labels Thai holidays separately from demand-inferred closures. `observed_by_university` is a supplied assumption supported by a later university calendar, not proof of building closure or occupancy in 2019.

The weekday-only 08:00–17:00 and 08:00–18:00 analyses remain the comparison cases. Additional `*_holiday_excluded` cases omit university-observed holiday labels from the operating windows, March calibration, April coverage denominators and scenario peaks/energy. March has no such holidays; April excludes 8, 12, 15 and 16 April. Expected April intervals become 648 and 720 respectively, compared with 792 and 880 for weekday-only cases.

All-hours readings remain available. Holiday energy is reclassified outside the assumed schedule, not counted as savings. The original baseline remains weekday-only for reproducibility.


# Lighting analytics: input preparation

Reuse the cleaned CU-BEMS files and the descriptive lighting baseline. This stage does not reclean data.
The population is 33 zones across seven floors, with March 6-31 calibration and April 2019 evaluation.
All timestamps are local Bangkok time; each row is a 15-minute interval start.

Run after `analytics/baseline/lighting_peak_baseline.ipynb`. The complete workflow is runnable with
`python analytics/run_lighting.py`. Methods, changes and verification are recorded in `../PROCESS.md`.

In [1]:
from pathlib import Path
import sys
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'analytics/lighting/config.json').is_file())
sys.path.insert(0, str(root / 'analytics/lighting'))
from IPython.display import display
from prepare_inputs import prepare_inputs, load_inputs, OUTPUT
intervals = prepare_inputs()
checked, manifest = load_inputs()
print(f'{len(checked):,} intervals; {checked[["floor", "zone"]].drop_duplicates().shape[0]} zones')
print(manifest['baseline_reconciliation'])
print('Outputs:', OUTPUT)
display(checked.head())

Prepared 177,408 intervals, 33 zones; saved baseline reconciled.


177,408 intervals; 33 zones
All 33 April operating-hour zone peaks reproduce saved baseline
Outputs: C:\Users\kiara_qi2p9x4\Documents\Classes_Y3\SBA-Final-Project\outputs\lighting


,Date,floor,zone,total_kW,total_coverage,light_kW,light_coverage,lux,lux_coverage,plug_kW,...,light_valid,lux_valid,paired_valid,light_interpolated_fraction,plug_interpolated_fraction,lux_interpolated_fraction,total_interpolated_fraction,repair_provenance_available,power_complete_without_repairs,paired_complete_without_repairs
0,2019-03-06 00:00:00,1,1,12.947333,1.0,12.302667,1.0,NaN,NaN,0.644667,...,True,False,False,0.0,0.0,0.0,0.0,True,True,False
1,2019-03-06 00:15:00,1,1,13.012667,1.0,12.318667,1.0,NaN,NaN,0.694000,...,True,False,False,0.0,0.0,0.0,0.0,True,True,False
2,2019-03-06 00:30:00,1,1,12.958667,1.0,12.342000,1.0,NaN,NaN,0.616667,...,True,False,False,0.0,0.0,0.0,0.0,True,True,False
3,2019-03-06 00:45:00,1,1,12.990000,1.0,12.381333,1.0,NaN,NaN,0.608667,...,True,False,False,0.0,0.0,0.0,0.0,True,True,False
4,2019-03-06 01:00:00,1,1,13.010000,1.0,12.400667,1.0,NaN,NaN,0.609333,...,True,False,False,0.0,0.0,0.0,0.0,True,True,False


## Common support and missingness

The original baseline uses at least 12 of 15 cleaned minutes. Strict scenarios require 15 power minutes;
control also requires 15 paired power/lux minutes. Common-minute sensitivity computes total, lighting and lux
on the same minute support. Missing lux disables action without deleting observed power.
Complete cleaned data can include interpolation; repair-free flags require the upstream repair log.
Schema absence, missing observations and measured zero remain distinct.

In [2]:
display(checked.groupby(['floor', 'zone']).agg(
    intervals=('Date', 'size'), has_lux=('has_lux_sensor', 'first'),
    complete_power_fraction=('power_complete', 'mean'),
    complete_paired_fraction=('paired_complete', 'mean'),
    repair_provenance=('repair_provenance_available', 'all')))
assert not checked.duplicated(['Date', 'floor', 'zone']).any()
assert len(checked[['floor', 'zone']].drop_duplicates()) == 33

intervals  has_lux  complete_power_fraction  \
floor zone                                                
1     1          5376    False                 1.000000   
      2          5376    False                 1.000000   
      3          5376    False                 1.000000   
      4          5376    False                 1.000000   
2     1          5376     True                 0.999442   
      2          5376     True                 0.999442   
      3          5376     True                 1.000000   
      4          5376     True                 0.999442   
3     1          5376     True                 0.833333   
      2          5376     True                 0.833333   
      3          5376    False                 1.000000   
      4          5376     True                 0.833333   
      5          5376     True                 0.833333   
4     1          5376     True                 0.011161   
      2          5376     True                 0.601376   
      3          5376    False                 1.000000   
      4          5376     True                 0.999442   
      5          5376     True                 0.999442   
5     1          5376     True                 0.999442   
      2          5376     True                 0.999442   
      3          5376    False                 1.000000   
      4          5376     True                 0.999442   
      5          5376     True                 0.999442   
6     1          5376     True                 0.999442   
      2          5376     True                 0.999442   
      3          5376    False                 1.000000   
      4          5376     True                 0.999442   
      5          5376     True                 0.999442   
7     1          5376     True                 0.981585   
      2          5376     True                 0.999442   
      3          5376    False                 0.988467   
      4          5376     True                 0.981399   
      5          5376     True                 0.981585   

            complete_paired_fraction  repair_provenance  
floor zone                                               
1     1                     0.000000               True  
      2                     0.000000               True  
      3                     0.000000               True  
      4                     0.000000               True  
2     1                     0.983631               True  
      2                     0.981213               True  
      3                     0.985863               True  
      4                     0.912016               True  
3     1                     0.816406               True  
      2                     0.725632               True  
      3                     0.000000               True  
      4                     0.808036               True  
      5                     0.809152               True  
4     1                     0.010789               True  
      2                     0.581473               True  
      3                     0.000000               True  
      4                     0.978051               True  
      5                     0.978051               True  
5     1                     0.967820               True  
      2                     0.946057               True  
      3                     0.000000               True  
      4                     0.686570               True  
      5                     0.965402               True  
6     1                     0.975446               True  
      2                     0.968378               True  
      3                     0.000000               True  
      4                     0.975074               True  
      5                     0.951637               True  
7     1                     0.949777               True  
      2                     0.978051               True  
      3                     0.000000               True  
      4                     0.95

## Schedule definitions and source integrity

Preserve the branch's primary weekday 08:00-17:00 assumption and compare 08:00-18:00 explicitly.
The original baseline retains 08:00-18:00. March/April denominators are 648/792 for the primary window and
720/880 for the extended window. The 17:00-18:00 transition is not measured vacancy.
Input hashes prevent stale shared tables from being consumed after source or configuration changes.

In [3]:
from schedules import schedule_settings, operating_index, label
import pandas as pd
settings = manifest['settings']
display(pd.DataFrame([dict(schedule_id=sid, window=label(s),
    march_intervals=len(operating_index(s, settings['calibration_start'], settings['evaluation_start'])),
    april_intervals=len(operating_index(s, settings['evaluation_start'], settings['evaluation_end'])))
    for sid, s in schedule_settings(settings)]))
print('Manifest source count:', len(manifest['sources']))
print('Shared table SHA256:', manifest['output_sha256'])
display(checked.loc[checked.month.eq('2019-04') & checked.observed_by_university,
    ['calendar_date', 'day_type', 'holiday_name', 'holiday_type', 'holiday_source']]
    .drop_duplicates().sort_values('calendar_date'))


,schedule_id,window,march_intervals,april_intervals
0,primary,08:00–17:00,648,792
1,extended,08:00–18:00,720,880
2,primary_holiday_excluded,08:00–17:00,648,648
3,extended_holiday_excluded,08:00–18:00,720,720


Manifest source count: 28
Shared table SHA256: 8de85b2b4f28e79e8af4bfde2be5c2d89ddb9209d0df4d5999b5b93ba8aef7e9


,calendar_date,day_type,holiday_name,holiday_type,holiday_source
2976,2019-04-06,weekend,Chakri Memorial Day,public,holidays package 0.108 (public)
3168,2019-04-08,closure,Chakri Memorial Day (in lieu),public,holidays package 0.108 (public)
3552,2019-04-12,closure,Songkran (additional special holiday),special,"Cabinet resolution, 12 Feb 2019"
3648,2019-04-13,weekend,Songkran Festival,public,holidays package 0.108 (public)
3744,2019-04-14,weekend,Songkran Festival,public,holidays package 0.108 (public)
3840,2019-04-15,closure,Songkran Festival,public,holidays package 0.108 (public)
3936,2019-04-16,closure,Songkran Festival (in lieu),public,holidays package 0.108 (public)
